### Support Vector Machine (SVM) — End-to-End Case Study

**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Goal:** Classify tumors as Malignant or Benign based on cell nucleus features.

---
#### Table of Contents
1. [Introduction & Theory](#theory)
2. [Import Libraries](#imports)
3. [Load & Explore Data](#eda)
4. [Data Preprocessing & Scaling](#preprocessing)
5. [SVM with Linear Kernel](#linear)
6. [SVM with RBF Kernel](#rbf)
7. [Kernel Comparison](#kernels)
8. [Hyperparameter Tuning (C & Gamma)](#tuning)
9. [Decision Boundary Visualization](#boundary)
10. [Final Evaluation & Model Comparison](#evaluation)
11. [Key Takeaways](#takeaways)

---
## 1. 📚 Introduction & Theory <a id='theory'></a>

### What is SVM?
A **Support Vector Machine** finds the **optimal hyperplane** that best separates classes by **maximizing the margin** between them.

```
        Class A (●)        Margin        Class B (■)
            ●                               ■
          ●   ●        ←— margin —→       ■   ■
            ●      ⬡ ─────────────── ⬡      ■
                 Support Vectors   Hyperplane
```

### Key Concepts

| Concept | Meaning |
|---------|----------|
| **Hyperplane** | Decision boundary separating classes |
| **Support Vectors** | Data points closest to the hyperplane (define the margin) |
| **Margin** | Distance between hyperplane and nearest support vectors — SVM maximizes this |
| **C (Regularization)** | Controls trade-off: small C = wider margin (more misclassifications ok); large C = narrow margin (fewer misclassifications) |
| **Kernel Trick** | Maps data to higher dimensions to make it linearly separable |

### Kernels
- **Linear**: `K(x, y) = xᵀy` — for linearly separable data
- **RBF (Gaussian)**: `K(x, y) = exp(-γ‖x-y‖²)` — most popular, handles non-linear data
- **Polynomial**: `K(x, y) = (γxᵀy + r)^d`
- **Sigmoid**: `K(x, y) = tanh(γxᵀy + r)`

### Hyperparameters
- **C**: Regularization — higher C = less regularization, tighter fit
- **gamma (γ)**: RBF kernel width — higher gamma = more complex boundary (can overfit)

### SVM vs Decision Tree vs Random Forest
| | SVM | Decision Tree | Random Forest |
|---|---|---|---|
| Works well with | High-dim, small datasets | Tabular data | Tabular data |
| Scaling needed | ✅ Yes | ❌ No | ❌ No |
| Interpretability | ❌ Low | ✅ High | ⚠️ Medium |
| Outlier sensitivity | ✅ Robust | ⚠️ Moderate | ✅ Robust |

---
### 2. Import Libraries <a id='imports'></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Datasets
from sklearn.datasets import load_breast_cancer

# Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.svm import SVC
from sklearn.decomposition import PCA

# Evaluation
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print(" All libraries imported successfully!")

 All libraries imported successfully!


---
## 3. Load & Explore Data <a id='eda'></a>

In [2]:
# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target   # 0 = Malignant, 1 = Benign
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f"Dataset shape : {df.shape}")
print(f"Features      : {len(data.feature_names)}")
print(f"\nTarget classes: {dict(df['diagnosis'].value_counts())}")
df.head()

Dataset shape : (569, 32)
Features      : 30

Target classes: {'Benign': 357, 'Malignant': 212}


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0,Malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0,Malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0,Malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0,Malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0,Malignant


In [ ]:
print("Statistical Summary:")
df.drop(columns=['target','diagnosis']).describe().round(3)

In [ ]:
# EDA visualizations
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Class distribution
colors = ['#e74c3c', '#2ecc71']
df['diagnosis'].value_counts().plot(kind='bar', ax=axes[0], color=colors, width=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Malignant', 'Benign'], rotation=0)
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()+3),
                     ha='center', fontsize=11)

# 2. Mean radius by diagnosis
df.boxplot(column='mean radius', by='diagnosis', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Mean Radius by Diagnosis', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
plt.sca(axes[1])
plt.title('Mean Radius by Diagnosis', fontsize=13, fontweight='bold')

# 3. Mean texture vs mean radius
for label, color, marker in [('Malignant', '#e74c3c', 'x'), ('Benign', '#2ecc71', 'o')]:
    mask = df['diagnosis'] == label
    axes[2].scatter(df[mask]['mean radius'], df[mask]['mean texture'],
                   c=color, label=label, alpha=0.6, marker=marker, s=40)
axes[2].set_xlabel('Mean Radius')
axes[2].set_ylabel('Mean Texture')
axes[2].set_title('Mean Radius vs Texture', fontsize=13, fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top correlated features with target
corr_with_target = df.drop(columns=['diagnosis']).corr()['target'].drop('target').abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
corr_with_target.head(15).plot(kind='barh', color='steelblue', alpha=0.8)
plt.xlabel('Absolute Correlation with Target', fontsize=11)
plt.title('Top 15 Features Correlated with Diagnosis', fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
### 4. Data Preprocessing & Scaling <a id='preprocessing'></a>

>  **Critical for SVM**: SVM is sensitive to feature scales. Always apply `StandardScaler` before training!

In [3]:
X = df.drop(columns=['target', 'diagnosis'])
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling — MANDATORY for SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # Only transform, never fit on test!

print(f"Training set : {X_train_scaled.shape}")
print(f"Test set     : {X_test_scaled.shape}")
print(f"\nBefore scaling — mean: {X_train.mean().mean():.2f}, std: {X_train.std().mean():.2f}")
print(f"After scaling  — mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")

Training set : (455, 30)
Test set     : (114, 30)

Before scaling — mean: 61.21, std: 33.98
After scaling  — mean: -0.0000, std: 1.0000


In [ ]:
# Why scaling matters — visual demo
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pd.DataFrame(X_train_scaled, columns=X.columns)[X.columns[:8]].boxplot(ax=axes[1])
X_train[X.columns[:8]].boxplot(ax=axes[0])

axes[0].set_title('Before Scaling (Features on different scales)', fontsize=12, fontweight='bold')
axes[1].set_title('After StandardScaler (Mean=0, Std=1)', fontsize=12, fontweight='bold')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
### 5.SVM with Linear Kernel <a id='linear'></a>

In [ ]:
# Train Linear SVM
svm_linear = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm_linear.fit(X_train_scaled, y_train)

# Predictions
y_pred_linear = svm_linear.predict(X_test_scaled)
y_prob_linear = svm_linear.predict_proba(X_test_scaled)[:, 1]

train_acc = accuracy_score(y_train, svm_linear.predict(X_train_scaled))
test_acc  = accuracy_score(y_test, y_pred_linear)
auc_score = roc_auc_score(y_test, y_prob_linear)

print("=" * 45)
print("SVM — LINEAR KERNEL (C=1.0)")
print("=" * 45)
print(f"Train Accuracy   : {train_acc*100:.2f}%")
print(f"Test  Accuracy   : {test_acc*100:.2f}%")
print(f"AUC Score        : {auc_score:.4f}")
print(f"Support Vectors  : {svm_linear.n_support_} (classes: 0 & 1)")
print(f"Total SVs        : {sum(svm_linear.n_support_)} / {len(X_train_scaled)} training samples")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_linear, target_names=['Malignant', 'Benign']))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_linear)
ConfusionMatrixDisplay(cm, display_labels=['Malignant', 'Benign']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Linear SVM — Confusion Matrix', fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob_linear)
axes[1].plot(fpr, tpr, color='steelblue', lw=2.5, label=f'Linear SVM (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Linear SVM — ROC Curve', fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

---
### 6.SVM with RBF Kernel <a id='rbf'></a>

In [ ]:
# Train RBF SVM
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svm_rbf.predict(X_test_scaled)
y_prob_rbf = svm_rbf.predict_proba(X_test_scaled)[:, 1]

train_acc_rbf = accuracy_score(y_train, svm_rbf.predict(X_train_scaled))
test_acc_rbf  = accuracy_score(y_test, y_pred_rbf)
auc_rbf       = roc_auc_score(y_test, y_prob_rbf)

print("=" * 45)
print("SVM — RBF KERNEL (C=1.0, gamma='scale')")
print("=" * 45)
print(f"Train Accuracy   : {train_acc_rbf*100:.2f}%")
print(f"Test  Accuracy   : {test_acc_rbf*100:.2f}%")
print(f"AUC Score        : {auc_rbf:.4f}")
print(f"Support Vectors  : {svm_rbf.n_support_}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_rbf, target_names=['Malignant', 'Benign']))

---
### 7. Kernel Comparison <a id='kernels'></a>

In [ ]:
# Compare all kernels
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
kernel_results = []

for k in kernels:
    model = SVC(kernel=k, C=1.0, probability=True, random_state=42)
    model.fit(X_train_scaled, y_train)
    pred  = model.predict(X_test_scaled)
    prob  = model.predict_proba(X_test_scaled)[:, 1]
    cv    = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()
    kernel_results.append({
        'Kernel': k,
        'Train Acc': accuracy_score(y_train, model.predict(X_train_scaled)),
        'Test Acc':  accuracy_score(y_test, pred),
        'AUC': roc_auc_score(y_test, prob),
        'CV (5-fold)': cv,
        'Num SVs': sum(model.n_support_)
    })

kernel_df = pd.DataFrame(kernel_results)
print(kernel_df.round(4).to_string(index=False))

In [ ]:
# Visual kernel comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(kernels))
width = 0.35

axes[0].bar(x_pos - width/2, kernel_df['Train Acc'], width, label='Train Accuracy',
            color='steelblue', alpha=0.85)
axes[0].bar(x_pos + width/2, kernel_df['Test Acc'], width, label='Test Accuracy',
            color='tomato', alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([k.upper() for k in kernels])
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.85, 1.02)
axes[0].set_title('Kernel Comparison — Accuracy', fontweight='bold', fontsize=13)
axes[0].legend()
for i, (tr, te) in enumerate(zip(kernel_df['Train Acc'], kernel_df['Test Acc'])):
    axes[0].text(i - width/2, tr + 0.002, f'{tr:.3f}', ha='center', fontsize=8)
    axes[0].text(i + width/2, te + 0.002, f'{te:.3f}', ha='center', fontsize=8)

# AUC comparison
axes[1].bar(x_pos, kernel_df['AUC'], color='mediumseagreen', alpha=0.85, width=0.5)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([k.upper() for k in kernels])
axes[1].set_ylabel('AUC Score')
axes[1].set_ylim(0.85, 1.02)
axes[1].set_title('Kernel Comparison — AUC Score', fontweight='bold', fontsize=13)
for i, v in enumerate(kernel_df['AUC']):
    axes[1].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
### 8. Hyperparameter Tuning (C & Gamma) <a id='tuning'></a>

**C** (Regularization parameter):
- Small C → Wider margin, more misclassifications allowed (underfitting risk)
- Large C → Narrow margin, fewer misclassifications (overfitting risk)

**Gamma** (RBF kernel width):
- Small γ → Large influence radius, smoother boundary
- Large γ → Small influence radius, complex boundary (overfitting risk)

In [ ]:
# Effect of C on accuracy
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_scores_C, test_scores_C = [], []

for C in C_values:
    m = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
    m.fit(X_train_scaled, y_train)
    train_scores_C.append(accuracy_score(y_train, m.predict(X_train_scaled)))
    test_scores_C.append(accuracy_score(y_test, m.predict(X_test_scaled)))

plt.figure(figsize=(10, 5))
plt.semilogx(C_values, train_scores_C, 'b-o', label='Train Accuracy', lw=2)
plt.semilogx(C_values, test_scores_C, 'r-s', label='Test Accuracy', lw=2)
plt.xlabel('C (log scale)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of C on SVM (RBF Kernel)', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.axvline(x=C_values[test_scores_C.index(max(test_scores_C))],
            color='green', linestyle='--', alpha=0.6,
            label=f'Best C={C_values[test_scores_C.index(max(test_scores_C))]}')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# C vs Gamma heatmap
C_range     = [0.1, 1, 10, 100]
gamma_range = [0.001, 0.01, 0.1, 1]
scores_grid = np.zeros((len(gamma_range), len(C_range)))

for i, gamma in enumerate(gamma_range):
    for j, C in enumerate(C_range):
        m = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        m.fit(X_train_scaled, y_train)
        scores_grid[i, j] = accuracy_score(y_test, m.predict(X_test_scaled))

plt.figure(figsize=(9, 6))
sns.heatmap(scores_grid,
            xticklabels=[f'C={c}' for c in C_range],
            yticklabels=[f'γ={g}' for g in gamma_range],
            annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5,
            annot_kws={'size': 11})
plt.title('SVM (RBF) — Test Accuracy: C vs Gamma', fontsize=13, fontweight='bold')
plt.xlabel('C (Regularization)')
plt.ylabel('Gamma (Kernel Width)')
plt.tight_layout()
plt.show()

In [ ]:
# GridSearchCV — best hyperparameters
param_grid = {
    'C':     [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print("Best Parameters :", grid_search.best_params_)
print(f"Best CV Score   : {grid_search.best_score_*100:.2f}%")

best_svm = grid_search.best_estimator_
best_pred = best_svm.predict(X_test_scaled)
best_prob = best_svm.predict_proba(X_test_scaled)[:, 1]
print(f"Test Accuracy   : {accuracy_score(y_test, best_pred)*100:.2f}%")
print(f"AUC Score       : {roc_auc_score(y_test, best_prob):.4f}")

---
### 9. Decision Boundary Visualization <a id='boundary'></a>

We use PCA to reduce 30 features → 2D for visualization.

In [ ]:
# Reduce to 2D using PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(scaler.fit_transform(X))
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# Decision boundary plotting function
def plot_decision_boundary(ax, model, X, y, title):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8, alpha=0.5)

    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn',
                        edgecolors='k', linewidths=0.4, s=30, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    return scatter

# Train SVMs on 2D PCA data
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [
    ('linear', 1.0, 'scale', 'SVM — Linear Kernel'),
    ('rbf',    1.0, 'scale', 'SVM — RBF Kernel (C=1)'),
    ('rbf',  100.0, 'auto',  'SVM — RBF Kernel (C=100, high overfit)'),
]

for ax, (kernel, C, gamma, title) in zip(axes, configs):
    m = SVC(kernel=kernel, C=C, gamma=gamma, random_state=42)
    m.fit(X_pca, y)
    acc = accuracy_score(y, m.predict(X_pca))
    plot_decision_boundary(ax, m, X_pca, y, f'{title}\nAcc={acc:.3f}')

plt.suptitle('SVM Decision Boundaries (2D PCA Projection)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 10. 📈 Final Evaluation & Model Comparison <a id='evaluation'></a>

In [ ]:
# Final confusion matrix for best model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_best = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm_best, display_labels=['Malignant', 'Benign']).plot(
    ax=axes[0], colorbar=False, cmap='Purples')
axes[0].set_title(f'Best SVM — Confusion Matrix\n{grid_search.best_params_}', fontweight='bold')

# Multi-model ROC comparison
models_roc = {
    'Linear SVM': svm_linear,
    'RBF SVM': svm_rbf,
    'Best SVM (Tuned)': best_svm
}
colors_roc = ['steelblue', 'tomato', 'purple']

for (name, model), color in zip(models_roc.items(), colors_roc):
    probs = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison — All SVM Models', fontweight='bold')
axes[1].legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Final Classification Report (Best SVM):")
print(classification_report(y_test, best_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# Cross-validation comparison of all models
final_models = {
    'SVM Linear': SVC(kernel='linear', C=1, probability=True, random_state=42),
    'SVM RBF': SVC(kernel='rbf', C=1, gamma='scale', probability=True, random_state=42),
    'SVM Poly': SVC(kernel='poly', C=1, probability=True, random_state=42),
    'Best SVM (Tuned)': best_svm
}

all_cv_scores = {}
summary = []

for name, model in final_models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring='accuracy')
    all_cv_scores[name] = cv_scores
    model.fit(X_train_scaled, y_train)
    test_a = accuracy_score(y_test, model.predict(X_test_scaled))
    auc_s  = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    summary.append({'Model': name, 'CV Mean': cv_scores.mean(),
                    'CV Std': cv_scores.std(), 'Test Acc': test_a, 'AUC': auc_s})

summary_df = pd.DataFrame(summary)
print("\n=== FINAL MODEL SUMMARY ===")
print(summary_df.round(4).to_string(index=False))

In [ ]:
# CV score distribution boxplot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].boxplot(all_cv_scores.values(), labels=all_cv_scores.keys(),
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_ylabel('CV Accuracy')
axes[0].set_title('10-Fold CV Distribution', fontweight='bold', fontsize=13)
axes[0].tick_params(axis='x', rotation=15)

# Summary bar chart
x = np.arange(len(summary_df))
w = 0.35
axes[1].bar(x - w/2, summary_df['Test Acc'], w, label='Test Accuracy', color='steelblue', alpha=0.85)
axes[1].bar(x + w/2, summary_df['AUC'], w, label='AUC Score', color='mediumseagreen', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df['Model'], rotation=15, ha='right')
axes[1].set_ylim(0.92, 1.01)
axes[1].set_ylabel('Score')
axes[1].set_title('Test Accuracy & AUC Comparison', fontweight='bold', fontsize=13)
axes[1].legend()
for i, row in summary_df.iterrows():
    axes[1].text(i - w/2, row['Test Acc'] + 0.001, f"{row['Test Acc']:.3f}", ha='center', fontsize=8)
    axes[1].text(i + w/2, row['AUC'] + 0.001, f"{row['AUC']:.3f}", ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Predict on a new sample
print("=" * 55)
print("PREDICT DIAGNOSIS FOR A NEW PATIENT SAMPLE")
print("=" * 55)

# Use a real sample from the dataset (index 10)
sample_idx = 10
new_sample = X.iloc[[sample_idx]]
true_label = y.iloc[sample_idx]

new_sample_scaled = scaler.transform(new_sample)
pred_label = best_svm.predict(new_sample_scaled)[0]
pred_prob  = best_svm.predict_proba(new_sample_scaled)[0]

label_map = {0: 'Malignant 🔴', 1: 'Benign 🟢'}
print(f"\nTrue Diagnosis    : {label_map[true_label]}")
print(f"Predicted         : {label_map[pred_label]}")
print(f"Malignant Prob    : {pred_prob[0]*100:.2f}%")
print(f"Benign Prob       : {pred_prob[1]*100:.2f}%")
correct = '✅ Correct!' if pred_label == true_label else '❌ Incorrect'
print(f"Prediction Result : {correct}")

---
### 11. Key Takeaways <a id='takeaways'></a>

### SVM Cheat Sheet

```
STEP 1: Scale features (StandardScaler) — ALWAYS
STEP 2: Choose kernel:
         Linear  → linearly separable / high-dim text data
         RBF     → default, works for most problems
         Poly    → image classification
STEP 3: Tune C and gamma via GridSearchCV
STEP 4: Evaluate with Accuracy, AUC, and CV scores
```

### When to use SVM?
- High-dimensional data (text, genomics)  
- Small to medium datasets (<10K samples)  
- Clear margin of separation exists  
- When you need robust performance with proper tuning  

### When NOT to use SVM?
-  Very large datasets (slow training: O(n²) to O(n³))  
- Lots of noise / overlapping classes  
- When you need model interpretability  

### Tuning Cheatsheet
| Symptom | Fix |
|---------|-----|
| Underfitting | Increase C; try RBF kernel |
| Overfitting | Decrease C; decrease gamma |
| Slow training | Reduce data size; use LinearSVC |
| Poor accuracy | Scale features; tune C & gamma |